# Figure 5-2: CellChat and final Figure 5 extensions

This self-contained notebook retains the all-gene CellChat analysis and generates the final Figure 5A-D source data and panels.


## 1. Prepare the all-gene CellChat input


In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import io as spio


METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
INPUTS = METHOD_DIR / "inputs"
OUTPUTS = METHOD_DIR / "outputs"

INPUT_H5AD = INPUTS / "adata_anno_cell_subtype_re.h5ad"
SUBTYPE_INPUT_H5AD = INPUTS / "subtype_full_lr_input.h5ad"
SUBTYPE_PAIRS_CSV = INPUTS / "subtype_all_pairs.csv"
CM_NODE_MAPPING_CSV = INPUTS / "cm_node_mapping_for_subtype_aggregation.csv"
EPI_SUBTYPES_CSV = INPUTS / "epi_subtypes.csv"
GROUP_KEY = "ccc_group"
SUBTYPE_KEY = "cell_subtype"


def read_required_csv(path: Path, required_cols: set[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")
    frame = pd.read_csv(path)
    if not required_cols.issubset(frame.columns):
        raise RuntimeError(f"{path} must contain {sorted(required_cols)}")
    return frame


def load_cm_node_mapping(valid_subtypes: set[str]) -> pd.DataFrame:
    mapping = read_required_csv(
        CM_NODE_MAPPING_CSV,
        {"CM", "reference_node_rank", "node"},
    ).copy()
    mapping["CM"] = mapping["CM"].astype(str)
    mapping["node"] = mapping["node"].astype(str)
    mapping = mapping[mapping["CM"].str.startswith("joint_")]
    mapping = mapping.drop_duplicates(["CM", "node"])
    if "node_in_adata" in mapping.columns:
        marked = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "1", "yes"])
        mapping = mapping[marked]
    return mapping[mapping["node"].isin(valid_subtypes)].copy()


def export_cellchat_input(adata_obj: ad.AnnData) -> None:
    output_dir = INPUTS / "cellchat_input"
    output_dir.mkdir(parents=True, exist_ok=True)
    spio.mmwrite(output_dir / "expression_genes_by_cells.mtx", adata_obj.X.T)
    pd.DataFrame({"gene": adata_obj.var_names}).to_csv(
        output_dir / "genes.tsv", sep="\t", index=False
    )
    pd.DataFrame({"cell": adata_obj.obs_names}).to_csv(
        output_dir / "cells.tsv", sep="\t", index=False
    )
    optional = [
        column
        for column in ["cell_type", "cell_subtype", "sample", "status"]
        if column in adata_obj.obs.columns
    ]
    metadata = adata_obj.obs[[GROUP_KEY] + optional].copy()
    metadata.insert(0, "cell", adata_obj.obs_names)
    metadata.to_csv(output_dir / "metadata.tsv", sep="\t", index=False)


def prepare_input(force: bool = False) -> None:
    required_exports = [
        SUBTYPE_INPUT_H5AD,
        INPUTS / "cellchat_input/expression_genes_by_cells.mtx",
        INPUTS / "cellchat_input/genes.tsv",
        INPUTS / "cellchat_input/cells.tsv",
        INPUTS / "cellchat_input/metadata.tsv",
    ]
    if all(path.exists() for path in required_exports) and not force:
        return
    if not INPUT_H5AD.exists():
        raise FileNotFoundError(f"Missing input AnnData: {INPUT_H5AD}")

    adata = ad.read_h5ad(INPUT_H5AD, backed="r")
    obs = adata.obs.copy()
    if SUBTYPE_KEY not in obs.columns:
        raise RuntimeError(f"{INPUT_H5AD} lacks obs['{SUBTYPE_KEY}']")
    obs[SUBTYPE_KEY] = obs[SUBTYPE_KEY].astype(str)

    all_subtypes = set(obs[SUBTYPE_KEY].unique())
    cm_mapping = load_cm_node_mapping(all_subtypes)
    cm_nodes = sorted(cm_mapping["node"].unique())
    epithelial = read_required_csv(EPI_SUBTYPES_CSV, {"Epi"})
    pair_table = read_required_csv(SUBTYPE_PAIRS_CSV, {"source", "target"})
    analysis_groups = sorted(
        (set(pair_table["source"].astype(str)) | set(pair_table["target"].astype(str)))
        & all_subtypes
    )
    selected_names = obs.index[obs[SUBTYPE_KEY].isin(analysis_groups)].tolist()

    if adata.raw is not None:
        result = adata.raw[selected_names, :].to_adata().copy()
        expression_source = ".raw normalized/log expression"
    else:
        result = adata[selected_names, :].to_memory().copy()
        expression_source = ".X expression"

    result.obs[GROUP_KEY] = result.obs[SUBTYPE_KEY].astype(str)
    result.obs["direction_class"] = np.where(
        result.obs[GROUP_KEY].str.startswith("Epi_"),
        "Epi",
        np.where(result.obs[GROUP_KEY].isin(cm_nodes), "CM_node", "Other_subtype"),
    )
    result.uns["cm_groups"] = sorted(cm_mapping["CM"].unique())
    result.uns["epi_groups"] = [
        name for name in epithelial["Epi"].astype(str) if name in all_subtypes
    ]
    result.uns["cm_nodes"] = cm_nodes
    result.uns["all_subtype_groups"] = analysis_groups
    result.uns["input_note"] = (
        "All-gene CellChat input without downsampling; grouped by cell_subtype; "
        f"expression source: {expression_source}."
    )
    result.write_h5ad(SUBTYPE_INPUT_H5AD)
    export_cellchat_input(result)

prepare_input(force=False)


## 2. Run CellChat and retain the two required analysis outputs


In [ ]:
%%script R
method_dir <- file.path(getwd(), "cellchat_r_all_genes")
input_dir <- file.path(method_dir, "inputs/cellchat_input")
output_dir <- file.path(method_dir, "outputs")
pairs_csv <- file.path(method_dir, "inputs/subtype_all_pairs.csv")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/cm_epi_communication_r_tmp_official_ccc")
Sys.setenv(R_LIBS_USER = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"), .libPaths()))
suppressPackageStartupMessages({
  library(CellChat)
  library(Matrix)
  library(future)
})

cellchat_workers <- as.integer(Sys.getenv("CELLCHAT_WORKERS", "8"))
if (is.na(cellchat_workers) || cellchat_workers < 1) {
  cellchat_workers <- 1
}
options(future.globals.maxSize = 100 * 1024^3)
future::plan("multisession", workers = cellchat_workers)

expr <- readMM(file.path(input_dir, "expression_genes_by_cells.mtx"))
genes <- read.delim(file.path(input_dir, "genes.tsv"), stringsAsFactors = FALSE)
cells <- read.delim(file.path(input_dir, "cells.tsv"), stringsAsFactors = FALSE)
meta <- read.delim(file.path(input_dir, "metadata.tsv"), stringsAsFactors = FALSE)
rownames(expr) <- make.unique(genes$gene)
colnames(expr) <- cells$cell
rownames(meta) <- meta$cell
meta$ccc_group <- factor(meta$ccc_group)

cellchat <- createCellChat(object = expr, meta = meta, group.by = "ccc_group")
cellchat@DB <- CellChatDB.human
cellchat <- subsetData(cellchat)
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)
cellchat <- computeCommunProb(cellchat, type = "triMean", nboot = 100, seed.use = 1337)
cellchat <- filterCommunication(cellchat, min.cells = 10)
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)
saveRDS(cellchat, file.path(output_dir, "cellchat_object.rds"))

all_communication <- subsetCommunication(cellchat)
pairs <- read.csv(pairs_csv, stringsAsFactors = FALSE)
selected_communication <- merge(
  all_communication,
  pairs[, c("source", "target")],
  by = c("source", "target")
)
write.csv(
  selected_communication,
  file.path(output_dir, "official_cellchat_subtype_lr_results.csv"),
  row.names = FALSE
)

future::plan("sequential")


## 3. Prepare the Epi_JUN-to-CM1 bubble-plot CSV


In [ ]:

METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
OUTPUT_DIR = METHOD_DIR / "outputs"
MAPPING_CSV = METHOD_DIR / "inputs/cm_node_mapping_for_subtype_aggregation.csv"
CELLCHAT_LR_CSV = OUTPUT_DIR / "official_cellchat_subtype_lr_results.csv"
BUBBLE_DIR = OUTPUT_DIR / "claim_support/epi_jun_cm1_bubble"
BUBBLE_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"

SOURCE = "Epi_JUN"
CM = "joint_01_sharedCM"
KEEP_LR_PAIRS = {
    "GDF15|TGFBR2",
    "MIF|CD74_CD44",
    "MIF|CD74_CXCR4",
    "SPP1|CD44",
    "SPP1|ITGA4_ITGB1",
    "SPP1|ITGAV_ITGB1",
    "SPP1|ITGAV_ITGB5",
    "TGFB1|TGFbR1_R2",
    "TNFSF9|TNFRSF9",
}


def load_cm_nodes() -> list[str]:
    mapping = pd.read_csv(MAPPING_CSV)
    if "node_in_adata" in mapping.columns:
        keep = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "t", "1", "yes"])
        mapping = mapping.loc[keep].copy()
    selected = mapping.loc[mapping["CM"].astype(str).eq(CM)].copy()
    selected = selected.sort_values("reference_node_rank")
    return selected["node"].astype(str).drop_duplicates().tolist()


def make_epi_jun_cm1_bubble_data() -> None:
    communication = pd.read_csv(CELLCHAT_LR_CSV)
    communication["source"] = communication["source"].astype(str)
    communication["target"] = communication["target"].astype(str)
    communication["ligand"] = communication["ligand"].astype(str)
    communication["receptor"] = communication["receptor"].astype(str)
    communication["pval"] = pd.to_numeric(communication["pval"], errors="coerce")
    communication["communication_weight"] = pd.to_numeric(
        communication["communication_weight"]
        if "communication_weight" in communication.columns
        else communication["prob"],
        errors="coerce",
    ).fillna(0)
    communication["lr_pair"] = communication["ligand"] + "|" + communication["receptor"]

    bubble_data = communication.loc[
        communication["source"].eq(SOURCE)
        & communication["target"].isin(load_cm_nodes())
        & communication["pval"].le(0.05)
        & communication["annotation"].astype(str).eq("Secreted Signaling")
        & communication["lr_pair"].isin(KEEP_LR_PAIRS)
    ].copy()
    bubble_data = bubble_data.sort_values(["target", "lr_pair"]).reset_index(drop=True)
    BUBBLE_DIR.mkdir(parents=True, exist_ok=True)
    bubble_data.to_csv(BUBBLE_CSV, index=False)


make_epi_jun_cm1_bubble_data()


## 4. Draw the retained Figure 5B bubble plot


In [ ]:
import matplotlib

matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


METHOD_DIR = Path.cwd() / "cellchat_r_all_genes"
BUBBLE_DIR = METHOD_DIR / "outputs/claim_support/epi_jun_cm1_bubble"
DATA_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"
OUTPUT_PDF = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_plot.pdf"


def plot_epi_jun_cm1_bubble() -> None:
    data = pd.read_csv(DATA_CSV)
    color_col = "prob" if "prob" in data.columns else "lr_probs"
    size_col = "communication_weight"
    data[color_col] = pd.to_numeric(data[color_col], errors="coerce").fillna(0)
    data[size_col] = pd.to_numeric(data[size_col], errors="coerce").fillna(0)
    data["lr_pair"] = data["lr_pair"].astype(str)
    data["target"] = data["target"].astype(str)

    target_order = data.groupby("target", observed=True)[size_col].sum().sort_values(ascending=False).index.tolist()
    pair_order = data.groupby("lr_pair", observed=True)[size_col].max().sort_values(ascending=True).index.tolist()
    x_map = {value: index for index, value in enumerate(target_order)}
    y_map = {value: index for index, value in enumerate(pair_order)}

    weights = data[size_col].to_numpy(dtype=float)
    max_weight = np.nanmax(weights) if len(weights) else 0
    sizes = 18 + 210 * np.sqrt(weights / max_weight) if max_weight > 0 else np.repeat(28, len(data))
    fig_width = min(max(7.5, 0.55 * len(target_order) + 3.0), 11.0)
    fig_height = min(max(4.35, 0.20 * len(pair_order) + 2.1), 7.6)

    fig, axis = plt.subplots(figsize=(fig_width, fig_height))
    scatter = axis.scatter(
        data["target"].map(x_map).to_numpy(),
        data["lr_pair"].map(y_map).to_numpy(),
        c=data[color_col].to_numpy(dtype=float),
        s=sizes,
        cmap="viridis",
        edgecolors="none",
        linewidths=0,
        alpha=0.9,
    )
    axis.set_xticks(range(len(target_order)))
    axis.set_xticklabels(target_order, rotation=45, ha="right", fontsize=7)
    axis.set_yticks(range(len(pair_order)))
    axis.set_yticklabels(pair_order, fontsize=6)
    axis.set_xlabel("CM component subtype", fontsize=8)
    axis.set_ylabel("Ligand-receptor pair", fontsize=8)
    axis.set_title("CellChat official: Epi_JUN to CM1", fontsize=10)
    axis.grid(True, axis="both", color="0.88", linewidth=0.5)
    axis.set_axisbelow(True)
    colorbar = fig.colorbar(scatter, ax=axis, fraction=0.025, pad=0.01)
    colorbar.set_label(color_col, fontsize=7)
    colorbar.ax.tick_params(labelsize=6)

    if np.any(weights > 0):
        legend_values = np.quantile(weights[weights > 0], [0.33, 0.66, 1.0])
        handles = [
            plt.scatter(
                [], [], s=18 + 210 * np.sqrt(value / max_weight),
                color="0.55", edgecolors="none", linewidths=0,
            )
            for value in legend_values
        ]
        axis.legend(
            handles,
            [f"{value:.3f}" for value in legend_values],
            title=size_col,
            loc="upper right",
            bbox_to_anchor=(1.0, 1.02),
            fontsize=6,
            title_fontsize=6,
            frameon=True,
        )
    fig.tight_layout()
    fig.savefig(OUTPUT_PDF, bbox_inches="tight")
    plt.close(fig)


plot_epi_jun_cm1_bubble()


## 5. Final Figure 5A-B: paired-tumour source data and plots

Both panels use the same 41 tumours with at least 20 Epi_JUN and 20 Epi_VIM cells. Figure 5A uses raw-count pseudobulk edgeR (`~ sample + state`); Figure 5B uses mean log-normalized SPP1 expression from the all-gene input.


In [ ]:
from scipy import sparse

PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
FINAL_SOURCE = METHOD_DIR / "outputs/final_figure_source_data"
FINAL_SOURCE.mkdir(parents=True, exist_ok=True)
RAW_JUN_VIM = PROJECT_ROOT / "ccrcc_cellnavi/data/ccrcc_epi_jun_to_epi_vim_raw_counts.h5ad"
STATES, MIN_CELLS = ["Epi_JUN", "Epi_VIM"], 20

adata_counts = ad.read_h5ad(RAW_JUN_VIM, backed="r")
obs = adata_counts.obs[["status", "sample", "series", "cell_subtype"]].astype(str)
tumour = obs.status.eq("tumor") & obs.cell_subtype.isin(STATES)
cell_counts = obs.loc[tumour].groupby(["sample", "cell_subtype"], observed=True).size().unstack(fill_value=0).reindex(columns=STATES, fill_value=0)
paired = cell_counts.index[(cell_counts >= MIN_CELLS).all(axis=1)].astype(str).tolist()
keep = tumour & obs["sample"].isin(paired)
cell_index = np.flatnonzero(keep.to_numpy())
selected = obs.iloc[cell_index].copy()
groups = [(sample, state) for sample in paired for state in STATES]
lookup = {group: index for index, group in enumerate(groups)}
group_index = np.asarray([lookup[pair] for pair in zip(selected["sample"], selected["cell_subtype"])])
matrix = adata_counts.X[cell_index, :]
if hasattr(matrix, "to_memory"): matrix = matrix.to_memory()
matrix = sparse.csr_matrix(matrix, dtype=np.float64)
indicator = sparse.csr_matrix((np.ones(len(group_index)), (group_index, np.arange(len(group_index)))), shape=(len(groups), len(group_index)))
pseudobulk = (indicator @ matrix).toarray()
n_cells = np.asarray(indicator.sum(axis=1)).ravel().astype(int)
sample_series = selected.drop_duplicates("sample").set_index("sample")["series"].to_dict()
metadata = pd.DataFrame({"library_id": [f"{s}__{g}" for s,g in groups], "sample": [s for s,_ in groups], "series": [sample_series[s] for s,_ in groups], "state": [g for _,g in groups], "status": "tumor", "n_cells": n_cells})
pd.DataFrame(pseudobulk.T, index=adata_counts.var_names.astype(str), columns=metadata.library_id).round().astype(np.int64).to_csv(FINAL_SOURCE / "Figure5AB_paired_pseudobulk_counts.csv.gz", compression="gzip")
metadata.to_csv(FINAL_SOURCE / "Figure5AB_paired_pseudobulk_metadata.csv", index=False)
adata_counts.file.close()

adata_log = ad.read_h5ad(SUBTYPE_INPUT_H5AD, backed="r")
log_obs = adata_log.obs[["status", "sample", "series", GROUP_KEY]].astype(str)
log_keep = log_obs.status.eq("tumor") & log_obs["sample"].isin(paired) & log_obs[GROUP_KEY].isin(STATES)
log_index = np.flatnonzero(log_keep.to_numpy())
values = adata_log.X[log_index, adata_log.var_names.get_loc("SPP1")]
if hasattr(values, "to_memory"): values = values.to_memory()
values = values.toarray().ravel() if sparse.issparse(values) else np.asarray(values).ravel()
spp1_long = log_obs.iloc[log_index][["sample", "series", GROUP_KEY]].copy(); spp1_long["SPP1"] = values
paired_spp1 = spp1_long.groupby(["sample", "series", GROUP_KEY], observed=True).SPP1.mean().unstack(GROUP_KEY).dropna(subset=STATES).reset_index()
paired_spp1["Epi_JUN_minus_Epi_VIM"] = paired_spp1.Epi_JUN - paired_spp1.Epi_VIM
paired_spp1.to_csv(FINAL_SOURCE / "Figure5B_paired_SPP1_expression_source_data.csv", index=False)
adata_log.file.close()
assert len(paired) == 41 and len(paired_spp1) == 41


In [ ]:
%%script R
suppressPackageStartupMessages(library(edgeR))
source_dir <- file.path(getwd(), "cellchat_r_all_genes/outputs/final_figure_source_data")
counts <- as.matrix(read.csv(gzfile(file.path(source_dir, "Figure5AB_paired_pseudobulk_counts.csv.gz")), row.names=1, check.names=FALSE)); storage.mode(counts) <- "numeric"
metadata <- read.csv(file.path(source_dir, "Figure5AB_paired_pseudobulk_metadata.csv"), check.names=FALSE, stringsAsFactors=FALSE)
stopifnot(identical(colnames(counts), metadata$library_id))
metadata$sample <- factor(metadata$sample); metadata$state <- relevel(factor(metadata$state), ref="Epi_VIM")
design <- model.matrix(~ sample + state, data=metadata); stopifnot(qr(design)$rank == ncol(design))
y <- DGEList(counts=round(counts)); keep <- filterByExpr(y, design=design, min.count=10)
y <- y[keep,,keep.lib.sizes=FALSE]; y <- calcNormFactors(y, method="TMM"); y <- estimateDisp(y, design, robust=TRUE)
fit <- glmQLFit(y, design, robust=TRUE); test <- glmQLFTest(fit, coef=which(colnames(design)=="stateEpi_JUN"))
result <- topTags(test, n=Inf, sort.by="none")$table; result$gene <- rownames(result)
result <- result[,c("gene","logFC","logCPM","F","PValue","FDR")]; result$contrast <- "Epi_JUN_minus_Epi_VIM"; result$n_paired_tumors <- length(unique(metadata$sample))
write.csv(result, gzfile(file.path(source_dir, "Figure5A_all_gene_paired_edgeR.csv.gz")), row.names=FALSE, quote=FALSE)


In [ ]:
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon

FIGURE_ROOT = PROJECT_ROOT / "review_revision_20260828_li/results"
COL = {"hi":"#B84A4A","neutral":"#6F7880","vim":"#2A9D8F","vim_dark":"#176F66","jun":"#E67E22","jun_dark":"#A94F0A","dark":"#263238","mid":"#66717B","grid":"#E5EAED"}
mpl.rcParams.update({"font.family":"Arial","font.size":8.5,"axes.labelsize":9,"xtick.labelsize":8,"ytick.labelsize":8,"axes.linewidth":0.75,"pdf.fonttype":42,"ps.fonttype":42})

def header(fig, letter, title, subtitle):
    fig.text(.035,.965,letter,fontsize=16,fontweight="bold",color=COL["dark"],va="top"); fig.text(.095,.965,title,fontsize=12.5,fontweight="bold",color=COL["dark"],va="top"); fig.text(.095,.910,subtitle,fontsize=8.5,color=COL["mid"],va="top")

candidates = ["SPP1","ANGPTL4","PROS1","TGFA","TNFSF9","MIF","GDF15","TGFB1","MDK"]
ligands = pd.read_csv(FINAL_SOURCE / "Figure5A_all_gene_paired_edgeR.csv.gz").query("gene in @candidates").sort_values("logFC",ascending=False).reset_index(drop=True)
out_a = FIGURE_ROOT / "figure5a_ranked_ligand_screen_all_fdr_muted"; out_a.mkdir(parents=True,exist_ok=True); ligands.to_csv(out_a / "Figure5A_ligand_screen_source_data.csv",index=False)
fig,ax=plt.subplots(figsize=(7.09,4.45)); fig.subplots_adjust(left=.19,right=.94,bottom=.17,top=.80); header(fig,"A","SPP1 is the top Epi_JUN-enriched candidate ligand","Paired tumor-only pseudobulk; 41 tumors")
for i,row in ligands.iterrows():
    is_spp1=row.gene=="SPP1"; color=COL["hi"] if is_spp1 else COL["neutral"]; ax.plot([0,row.logFC],[i,i],color=color,lw=3 if is_spp1 else 1.8,solid_capstyle="butt"); ax.scatter(row.logFC,i,s=82 if is_spp1 else 50,color=color,edgecolor="white",linewidth=.65,zorder=3); ax.text(row.logFC+(.025 if row.logFC>=0 else -.025),i,f"FDR {row.FDR:.1e}",va="center",ha="left" if row.logFC>=0 else "right",color=color,fontsize=7.3,fontweight="bold" if is_spp1 else "normal")
ax.axvline(0,color="#59636A",lw=.9); ax.set_yticks(range(len(ligands))); ax.set_yticklabels(ligands.gene); ax.invert_yaxis(); ax.set_xlim(min(-.66,ligands.logFC.min()-.16),max(.78,ligands.logFC.max()+.16)); ax.set_xlabel("Epi_JUN − Epi_VIM log2 fold change")
ax.text(.01,1.025,"←  Epi_VIM higher",transform=ax.transAxes,color=COL["neutral"],fontweight="bold",ha="left"); ax.text(.99,1.025,"Epi_JUN higher  →",transform=ax.transAxes,color=COL["neutral"],fontweight="bold",ha="right"); ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x",color=COL["grid"],lw=.65)
fig.savefig(out_a / "Figure5A_ranked_ligand_screen_all_FDR_muted.pdf",dpi=450,bbox_inches="tight"); plt.close(fig)

data = pd.read_csv(FINAL_SOURCE / "Figure5B_paired_SPP1_expression_source_data.csv").sort_values("Epi_JUN_minus_Epi_VIM").reset_index(drop=True); test=wilcoxon(data.Epi_JUN,data.Epi_VIM)
out_b=FIGURE_ROOT / "figure5b_paired_spp1_significance_gray_linetype_compact_width_groups"; out_b.mkdir(parents=True,exist_ok=True); data.to_csv(out_b / "Figure5B_paired_SPP1_expression_source_data.csv",index=False)
fig,ax=plt.subplots(figsize=(6.65,4.55)); fig.subplots_adjust(left=.13,right=.95,bottom=.16,top=.80); header(fig,"B","SPP1 expression is higher in Epi_JUN than Epi_VIM",f"Paired tumour-only comparison across {len(data)} tumours"); xl,xr=.08,.92
for _,row in data.iterrows():
    up=row.Epi_JUN_minus_Epi_VIM>0; ax.plot([xl,xr],[row.Epi_VIM,row.Epi_JUN],color="#929CA2",lw=.95 if up else 1.25,linestyle="-" if up else (0,(4,2.4)),alpha=.56 if up else .95)
ax.scatter(np.full(len(data),xl),data.Epi_VIM,s=28,color=COL["vim"],edgecolor="white",linewidth=.55,zorder=3); ax.scatter(np.full(len(data),xr),data.Epi_JUN,s=31,color=COL["jun"],edgecolor="white",linewidth=.55,zorder=3)
ax.plot([xl-.1,xl+.1],[data.Epi_VIM.median()]*2,color=COL["vim_dark"],lw=4); ax.plot([xr-.1,xr+.1],[data.Epi_JUN.median()]*2,color=COL["jun_dark"],lw=4); by=max(data.Epi_VIM.max(),data.Epi_JUN.max())+.10; bh=.07; ax.plot([xl,xl,xr,xr],[by,by+bh,by+bh,by],color=COL["dark"],lw=.9); ax.text((xl+xr)/2,by+bh+.035,f"Paired Wilcoxon $P = {test.pvalue:.2e}$",ha="center")
fig.legend(handles=[Line2D([0],[0],color="#929CA2",label="Increase"),Line2D([0],[0],color="#929CA2",linestyle=(0,(4,2.4)),label="Decrease")],loc="upper left",bbox_to_anchor=(.095,.865),frameon=False,ncol=2); ax.set_xlim(-.18,1.18); ax.set_ylim(.10,by+.34); ax.set_xticks([xl,xr]); ax.set_xticklabels(["Epi_VIM","Epi_JUN"],fontsize=10.5,fontweight="bold"); ax.set_ylabel("Mean log-normalized SPP1 expression"); ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y",color=COL["grid"],lw=.65)
fig.savefig(out_b / "Figure5B_paired_SPP1_change_paired_Wilcoxon_gray_solid_dashed_compact_width_groups.pdf",dpi=450,bbox_inches="tight"); plt.close(fig)


## 6. Final Figure 5C: tumour-only balanced all-gene CellChat

The all-gene CellChat object generated above is restricted to Epi_JUN, Epi_VIM and the ten CM1 nodes. Up to 100 cells are sampled per `sample × group` stratum before recomputing candidate-ligand communication probabilities.


In [ ]:
%%script R
Sys.setenv(R_LIBS_USER="/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"),.libPaths()))
suppressPackageStartupMessages({library(CellChat); library(future)})
root <- "/mnt/disk18t/lr_xcy/riku/kirc_205"
source_rds <- file.path(getwd(),"cellchat_r_all_genes/outputs/cellchat_object.rds")
out_dir <- file.path(root,"review_revision_20260828_li/results/figure5c_cellchat_all_genes/tumor_only_cellchat")
dir.create(out_dir,recursive=TRUE,showWarnings=FALSE)
cm1_nodes <- c("T_IL7R","T_GNLY","T_KLRB1","T_FOXP3","Mast_FYN_high","S_DCN","T_TOX","T_GZMK","NK_KLRF1","NK_XCL1")
source_states <- c("Epi_JUN","Epi_VIM"); groups_use <- c(source_states,cm1_nodes)
candidate_ligands <- c("SPP1","MIF","GDF15","TGFB1","TNFSF9","ANGPTL4","PROS1","TGFA","MDK")
cap <- 100L; seed <- 20260820L; set.seed(seed)
options(future.globals.maxSize=30*1024^3); future::plan("multicore",workers=4L)
source_object <- readRDS(source_rds); meta <- source_object@meta
meta$ccc_group <- as.character(meta$ccc_group); meta$sample <- as.character(meta$sample); meta$status <- as.character(meta$status)
eligible <- rownames(meta)[meta$status=="tumor" & meta$ccc_group %in% groups_use]
eligible_meta <- meta[eligible,,drop=FALSE]
strata <- split(rownames(eligible_meta),interaction(eligible_meta$ccc_group,eligible_meta$sample,drop=TRUE))
selected <- sort(unique(unlist(lapply(strata,function(cells) if(length(cells)<=cap) cells else sample(cells,cap)),use.names=FALSE)))
selection <- eligible_meta[selected,c("ccc_group","sample","status"),drop=FALSE]; selection$cell <- rownames(selection)
write.csv(selection[,c("cell","ccc_group","sample","status")],gzfile(file.path(out_dir,"tumor_only_balanced_cells.csv.gz")),row.names=FALSE)
data_use <- source_object@data[,selected,drop=FALSE]; meta_use <- selection[,c("ccc_group","sample","status"),drop=FALSE]; rm(source_object); invisible(gc())
cellchat_balanced <- createCellChat(object=data_use,meta=meta_use,group.by="ccc_group")
cellchat_balanced@DB <- CellChatDB.human; cellchat_balanced <- subsetData(cellchat_balanced)
cellchat_balanced <- identifyOverExpressedGenes(cellchat_balanced); cellchat_balanced <- identifyOverExpressedInteractions(cellchat_balanced)
lr_use <- cellchat_balanced@LR$LRsig[cellchat_balanced@LR$LRsig$ligand %in% candidate_ligands & cellchat_balanced@LR$LRsig$annotation=="Secreted Signaling",,drop=FALSE]
cellchat_balanced <- computeCommunProb(cellchat_balanced,type="triMean",LR.use=lr_use,population.size=FALSE,distance.use=FALSE,nboot=100,seed.use=seed)
cellchat_balanced <- filterCommunication(cellchat_balanced,min.cells=10)
communication <- subsetCommunication(cellchat_balanced); communication$source <- as.character(communication$source); communication$target <- as.character(communication$target)
focus <- communication[communication$source %in% source_states & communication$target %in% cm1_nodes & communication$annotation=="Secreted Signaling",,drop=FALSE]
write.csv(communication,file.path(out_dir,"tumor_only_balanced_cellchat_all_communication.csv"),row.names=FALSE)
write.csv(focus,file.path(out_dir,"tumor_only_epi_jun_vim_to_cm1_secreted.csv"),row.names=FALSE)
saveRDS(cellchat_balanced,file.path(out_dir,"tumor_only_balanced_cellchat_object.rds"),compress=FALSE)
future::plan("sequential")


In [ ]:
from matplotlib.patches import FancyArrowPatch

OUT_C = FIGURE_ROOT / "figure5c_cellchat_all_genes"
CELLCHAT_CSV = OUT_C / "tumor_only_cellchat/tumor_only_epi_jun_vim_to_cm1_secreted.csv"
CM1_ORDER = ["T_IL7R","T_GNLY","T_KLRB1","T_FOXP3","Mast_FYN_high","S_DCN","T_TOX","T_GZMK","NK_KLRF1","NK_XCL1"]
PAIR_ORDER = ["SPP1|CD44","SPP1|ITGA4_ITGB1","SPP1|ITGAV_ITGB1","SPP1|ITGAV_ITGB5"]
ROUTE_COLORS = {"vim":"#2A9D8F","jun":"#E67E22","cm1":"#3D6D9C","spp1":"#C94F49","dark":"#263238","mid":"#66717B","gray":"#9AA3AA"}

raw = pd.read_csv(CELLCHAT_CSV); raw["lr_pair"] = raw.ligand.astype(str)+"|"+raw.receptor.astype(str); raw=raw[raw.lr_pair.isin(PAIR_ORDER)]
grouped=raw.groupby(["source","target","lr_pair"],observed=True).prob.sum().rename("prob").reset_index()
grid=pd.MultiIndex.from_product([["Epi_JUN","Epi_VIM"],CM1_ORDER,PAIR_ORDER],names=["source","target","lr_pair"]).to_frame(index=False)
grouped=grid.merge(grouped,how="left").fillna({"prob":0}); wide=grouped.pivot(index=["target","lr_pair"],columns="source",values="prob").reset_index(); wide.columns.name=None
wide["max_probability"]=wide[["Epi_JUN","Epi_VIM"]].max(axis=1); wide["log2_jun_vim_probability_ratio"]=np.log2((wide.Epi_JUN+1e-4)/(wide.Epi_VIM+1e-4))
wide.to_csv(OUT_C / "tumor_only_cellchat_differential_bubble_source_data.csv",index=False)
data=wide.copy(); data["receiver_group"]=np.where(data.target.str.startswith(("T_","NK_")),"T/NK CM1 states",np.where(data.target.eq("S_DCN"),"Stromal DCN","Mast FYN-high"))
route=data.groupby(["lr_pair","receiver_group"])[["Epi_JUN","Epi_VIM"]].sum().reset_index(); route.to_csv(OUT_C / "Figure5C_all_genes_aggregated_CellChat_routes.csv",index=False)
totals=data[["Epi_JUN","Epi_VIM"]].sum(); ratio=float(totals.Epi_JUN/totals.Epi_VIM)

def arrow(ax,start,end,color,width,linestyle="-",shrink_a=7,shrink_b=7):
    ax.add_patch(FancyArrowPatch(start,end,connectionstyle="arc3,rad=0",arrowstyle="-|>",mutation_scale=9,lw=width,color=color,alpha=1,linestyle=linestyle,shrinkA=shrink_a,shrinkB=shrink_b,capstyle="butt",joinstyle="miter",zorder=1))
def score_label(ax,start,end,value,color,fraction=.5,offset=(0,0)):
    x=start[0]+fraction*(end[0]-start[0])+offset[0]; y=start[1]+fraction*(end[1]-start[1])+offset[1]; ax.text(x,y,f"{value:.3f}",ha="center",va="center",color=color,fontsize=7.4,fontweight="bold",bbox={"boxstyle":"square,pad=0.12","facecolor":"white","edgecolor":"none","alpha":.88},zorder=6)

fig,ax=plt.subplots(figsize=(7.09,4.65)); fig.subplots_adjust(left=.025,right=.98,bottom=.08,top=.80); header(fig,"C","Epi_JUN supplies a stronger predicted SPP1 route to CM1 states","Tumour-only balanced CellChat; labels show aggregated probability and widths are scaled within stage")
fig.legend(handles=[Line2D([0],[0],color=ROUTE_COLORS["jun"],lw=2.6,label="Epi_JUN total"),Line2D([0],[0],color=ROUTE_COLORS["vim"],lw=2.6,linestyle="--",label="Epi_VIM total"),Line2D([0],[0],color=ROUTE_COLORS["spp1"],lw=2.2,label="JUN: SPP1→receptor"),Line2D([0],[0],color=ROUTE_COLORS["cm1"],lw=2.2,label="JUN: receptor→receiver")],loc="upper left",bbox_to_anchor=(.085,.845),frameon=False,ncol=4,fontsize=7.1)
source_pos={"Epi_JUN":(.09,.67),"Epi_VIM":(.09,.33)}; spp1_pos=(.34,.50); receptor_pos={"SPP1|CD44":(.56,.79),"SPP1|ITGA4_ITGB1":(.56,.59),"SPP1|ITGAV_ITGB1":(.56,.39),"SPP1|ITGAV_ITGB5":(.56,.19)}; receptor_label={"SPP1|CD44":"CD44","SPP1|ITGA4_ITGB1":"α4β1","SPP1|ITGAV_ITGB1":"αVβ1","SPP1|ITGAV_ITGB5":"αVβ5"}; receiver_pos={"T/NK CM1 states":(.82,.70),"Stromal DCN":(.82,.37),"Mast FYN-high":(.82,.16)}
for source,pos in source_pos.items():
    color=ROUTE_COLORS["jun"] if source=="Epi_JUN" else ROUTE_COLORS["vim"]; arrow(ax,pos,spp1_pos,color,2.2+6*totals[source]/totals.max(),"--" if source=="Epi_VIM" else "-",13,15); ax.scatter(*pos,s=620,color=color,edgecolor="none",zorder=4); ax.text(*pos,source.replace("Epi_","Epi\n"),ha="center",va="center",color="white",fontsize=10,fontweight="bold",zorder=5); score_label(ax,pos,spp1_pos,totals[source],color,.48,(0,.035 if source=="Epi_JUN" else -.035))
ax.scatter(*spp1_pos,s=760,color=ROUTE_COLORS["spp1"],edgecolor="none",zorder=4); ax.text(*spp1_pos,"SPP1",ha="center",va="center",color="white",fontsize=11,fontweight="bold",zorder=5); ax.text(.205,.91,f"{ratio:.2f}× stronger total output",ha="center",color=ROUTE_COLORS["spp1"],fontweight="bold")
receptor_totals=data.groupby("lr_pair").Epi_JUN.sum()
for pair,pos in receptor_pos.items():
    arrow(ax,spp1_pos,pos,ROUTE_COLORS["spp1"],1.1+5.5*receptor_totals[pair]/receptor_totals.max(),"-",15,11); ax.scatter(*pos,s=370,color="white",edgecolor=ROUTE_COLORS["spp1"],linewidth=1.7,zorder=4); ax.text(*pos,receptor_label[pair],ha="center",va="center",color=ROUTE_COLORS["dark"],fontsize=9,fontweight="bold",zorder=5); offsets={"SPP1|CD44":(-.008,.022),"SPP1|ITGA4_ITGB1":(-.005,.024),"SPP1|ITGAV_ITGB1":(-.005,-.024),"SPP1|ITGAV_ITGB5":(-.005,-.024)}; score_label(ax,spp1_pos,pos,receptor_totals[pair],ROUTE_COLORS["spp1"],.52,offsets[pair])
positive_route=route[route.Epi_JUN>0]; max_route=positive_route.Epi_JUN.max()
for _,row in positive_route.iterrows():
    start,end=receptor_pos[row.lr_pair],receiver_pos[row.receiver_group]; arrow(ax,start,end,ROUTE_COLORS["cm1"],.9+4*row.Epi_JUN/max_route,"-",11,9); specs={("SPP1|CD44","T/NK CM1 states"):(.48,(0,.025)),("SPP1|CD44","Stromal DCN"):(.56,(.012,.016)),("SPP1|CD44","Mast FYN-high"):(.54,(.012,-.025)),("SPP1|ITGA4_ITGB1","T/NK CM1 states"):(.47,(0,-.026)),("SPP1|ITGAV_ITGB1","Stromal DCN"):(.48,(0,.024)),("SPP1|ITGAV_ITGB5","Stromal DCN"):(.48,(0,-.025))}; fraction,offset=specs.get((row.lr_pair,row.receiver_group),(.5,(0,0))); score_label(ax,start,end,row.Epi_JUN,ROUTE_COLORS["cm1"],fraction,offset)
for receiver,pos in receiver_pos.items():
    ax.scatter(*pos,s=260,color=ROUTE_COLORS["cm1"] if receiver=="T/NK CM1 states" else ROUTE_COLORS["gray"],edgecolor="none",zorder=4); ax.text(pos[0]+.035,pos[1],receiver,ha="left",va="center",color=ROUTE_COLORS["dark"],fontsize=8.7,fontweight="bold")
ax.text(.56,.94,"SPP1 receptors",ha="center",color=ROUTE_COLORS["mid"],fontweight="bold"); ax.text(.86,.94,"Receiver groups",ha="center",color=ROUTE_COLORS["mid"],fontweight="bold"); ax.text(.56,.035,"Source labels show total output; downstream labels show Epi_JUN route probability",ha="center",color=ROUTE_COLORS["mid"],fontsize=7.4); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
fig.savefig(OUT_C / "Figure5C_all_genes_SPP1_route_schematic_straight_node_edge_aligned_no_white_outline_opaque_edge_scores_vim_dashed_legend.pdf",bbox_inches="tight"); plt.close(fig)


## 7. Final Figure 5D: spatial SPP1-receptor hotspot source data and map

The fixed Tangram CM1/Epi_JUN spot scores are aligned to each spatial expression matrix. SPP1 and receptor gene-set scores are z-scored within section and converted to percentile ranks before defining top-quartile hotspots.


In [ ]:
import math
import scanpy as sc
from scipy import stats as scipy_stats

SPATIAL_ROOT = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/spatial")
TANGRAM_TABLES = SPATIAL_ROOT / "cm_assignment_analysis_method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells/tables"
SPATIAL_H5AD = SPATIAL_ROOT / "cm_assignment_analysis/assign_nonepi_cm/mendeley_nonepi_cm_h5ad_9x9KL"
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"
SECTIONS = ["h46t","j38t","r114t","r29t","r51t","s15t","x49t","x98t","y12t","y27t","y7t","z43t"]
RECEPTORS = ["CD44","ITGAV","ITGB1","ITGA4","ITGA5","ITGB5"]
OUT_D = FIGURE_ROOT / "figure5d_spatial_hotspot_maps_background_legend"; OUT_D.mkdir(parents=True,exist_ok=True)

def percentile(values):
    return pd.Series(values).rank(method="average",pct=True).to_numpy()
def gene_score(adata, genes):
    present=[gene for gene in genes if gene in adata.var_names]
    if not present: return np.full(adata.n_obs,np.nan)
    x=adata[:,present].X; x=x.toarray() if sparse.issparse(x) else np.asarray(x); sd=np.nanstd(x,axis=0); sd[sd==0]=np.nan
    return np.nanmean((x-np.nanmean(x,axis=0))/sd,axis=1)

spot_tables=[]; section_rows=[]
for sample in SECTIONS:
    score_path=TANGRAM_TABLES / f"{sample}_{METHOD}_spot_scores.csv"; h5ad_path=SPATIAL_H5AD / f"adata_{sample}_CM.h5ad"
    score=pd.read_csv(score_path,index_col=0); adata_sp=sc.read_h5ad(h5ad_path); common=adata_sp.obs_names.intersection(score.index); adata_sp=adata_sp[common].copy(); score=score.reindex(common).copy()
    for column in ["array_row","array_col"]:
        if column not in score and column in adata_sp.obs: score[column]=adata_sp.obs[column].to_numpy()
    score["CM1_pct"]=percentile(score["CMact__joint_01_sharedCM"]); score["Epi_JUN_pct"]=percentile(score["EPIfrac__Epi_JUN"]); score["JUN_CM1_double_high"]=score[["CM1_pct","Epi_JUN_pct"]].min(axis=1)
    score["SPP1_ligand_score"]=gene_score(adata_sp,["SPP1"]); score["SPP1_receptor_score"]=gene_score(adata_sp,RECEPTORS)
    score["SPP1_ligand_pct"]=percentile(score["SPP1_ligand_score"]); score["SPP1_receptor_pct"]=percentile(score["SPP1_receptor_score"]); score["SPP1_LR_double_high"]=score[["SPP1_ligand_pct","SPP1_receptor_pct"]].min(axis=1)
    score["sample"]=sample; score["spot_id"]=score.index; rho,p=scipy_stats.spearmanr(score.JUN_CM1_double_high,score.SPP1_LR_double_high,nan_policy="omit"); section_rows.append({"sample":sample,"n_spots":len(score),"spearman_rho":rho,"p_value":p}); spot_tables.append(score.reset_index(drop=True))
all_spots=pd.concat(spot_tables,ignore_index=True); spatial_summary=pd.DataFrame(section_rows)
all_spots.to_csv(FINAL_SOURCE / "Figure5D_spatial_hotspot_spot_source_data.csv.gz",index=False,compression="gzip"); spatial_summary.to_csv(FINAL_SOURCE / "Figure5D_spatial_hotspot_section_summary.csv",index=False)

data=all_spots[all_spots.sample.eq("x98t")].copy(); niche=data.JUN_CM1_double_high.to_numpy()>=.75; route=data.SPP1_LR_double_high.to_numpy()>=.75; overlap=niche & route
fig,axes=plt.subplots(1,3,figsize=(7.09,3.60)); fig.subplots_adjust(left=.025,right=.98,bottom=.08,top=.70,wspace=.04); n_positive=int((spatial_summary.spearman_rho>0).sum())
header(fig,"D","SPP1-receptor hotspots fall within JUN–CM1-rich tissue regions",f"Representative x98t section; top-quartile hotspots; section-level association is positive in {n_positive}/{len(spatial_summary)} sections")
hotspot_colors={"background":"#E8ECEF","niche":"#7651A8","route":"#C94F49","overlap":"#5B2E73"}
fig.legend(handles=[Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["background"],markeredgecolor="none",label="Background spots"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["niche"],markeredgecolor="none",label="JUN–CM1 hotspot"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["route"],markeredgecolor="none",label="SPP1–receptor hotspot"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["overlap"],markeredgecolor="none",label="Overlap hotspot")],loc="upper center",bbox_to_anchor=(.52,.825),frameon=False,ncol=4,fontsize=7.5)
x=data.array_col.to_numpy(float); y=-data.array_row.to_numpy(float)*math.sqrt(3)
for ax,mask,color,title in zip(axes,[niche,route,overlap],[hotspot_colors["niche"],hotspot_colors["route"],hotspot_colors["overlap"]],["JUN–CM1 hotspot","SPP1–receptor route",f"Spatial overlap\n{int(overlap.sum())} spots"]):
    ax.scatter(x,y,s=4,color=hotspot_colors["background"],linewidth=0,alpha=.85,rasterized=True); ax.scatter(x[mask],y[mask],s=9,color=color,linewidth=0,alpha=.98,rasterized=True); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title,color=color,fontsize=9.5,fontweight="bold",pad=5)
fig.savefig(OUT_D / "Figure5D_spatial_hotspot_maps_with_background_legend.pdf",dpi=450,bbox_inches="tight"); plt.close(fig)
